# W02 tutorial — the Growth-of-£1 Desk Tool

**ACFI20009 Financial Data — 26/27**  
Dr Zeming Li, University of Bristol

**This week you build the Growth-of-£1 Desk Tool:** a small tested pipeline plus the client exhibit it produces: load prices, build a price table with one column per ticker, compound £1, plot it, and guard the whole thing with assertions you write before you trust it.

Work straight down the notebook. Each section explains what we are doing and why, then does it. By the last cell the artefact exists and runs.

> ### 🤖 Working with Gemini in this notebook
>
> Colab has Gemini built in. You are **encouraged** to use it — reading and
> directing AI-written code is part of the craft this unit teaches.
>
> **One hard rule:** never paste rows of CRSP data into Gemini (or any other
> AI tool). The licence forbids it. Describe the *schema* instead — column
> names, dtypes, what a row means — which is all an assistant needs to write
> correct code.

## Setup

Run these two cells once, top to bottom, before anything else.

In [ ]:
# Nothing to install: this notebook uses only what Colab already ships
# (pandas, numpy, matplotlib, statsmodels, scipy, sklearn, pyarrow).

In [ ]:
# --- Data -------------------------------------------------------------------
# The unit's datasets are distributed through Blackboard. Download the
# `ACFI20009_data` folder once, put it in your own Google Drive, and this cell
# will find it. Nothing licensed is ever stored in the public repository, and
# the data never leaves your personal Drive.
from pathlib import Path

try:                                    # running on Colab
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/ACFI20009/data")
except ModuleNotFoundError:             # running locally (convenor)
    DATA = Path("../../data")

assert DATA.exists(), (
    f"Data folder not found at {DATA}.\n"
    "Download it from Blackboard and place it at MyDrive/ACFI20009/data"
)
print("Data:", DATA, "|", len(list(DATA.glob('*.parquet'))), "parquet files")

---

In [ ]:
import numpy as np, pandas as pd
pd.options.display.float_format = "{:.4f}".format

In [ ]:
pd.read_parquet(DATA / "stocks_daily.parquet").head()

If you see five rows of prices, you have passed the gate. If not, raise your hand now — the first half hour of this session exists to fix it. The most common cause by far is that `stocks_daily.parquet` is not directly inside the `data` folder in your Google Drive (for example, unzipping added an extra folder level): check with

In [ ]:
DATA, (DATA / "stocks_daily.parquet").exists()

The path should end in `MyDrive/ACFI20009/data` and the second value should be `True`. If it is `False`, `stocks_daily.parquet` is not directly inside that folder: check in Drive that the unzipped `data` folder sits at `MyDrive/ACFI20009/data`, then run the Setup cells again. Every notebook in this unit reads its files through `DATA`.

### 1. Load and *look*

In [ ]:
stocks = pd.read_parquet(DATA / "stocks_daily.parquet")
stocks.dtypes

Dtypes first, every time. This snapshot already stores dates as `datetime64`, prices as `float64`, volume as `int64` (whole shares) and tickers as `object` (strings) — you check, rather than convert, and the habit of checking is what carries over to files that are not so clean. Then the shape of the thing:

In [ ]:
stocks.shape, stocks["ticker"].unique()

Seven tickers, one row per (date, ticker) pair — so each date appears on several rows, once for each ticker.

### 2. Select, index, slice

Rows by condition, columns by name:

In [ ]:
spy = stocks.loc[stocks["ticker"] == "SPY", ["date", "adj_close"]]
spy = spy.set_index("date").sort_index()
spy.head(3)

Two slicing worlds — always know which one you are in:

In [ ]:
spy.iloc[0:3]            # by POSITION: rows 0,1,2 — half-open

In [ ]:
spy.loc["2020-03-20":"2020-03-23"]   # by LABEL: BOTH endpoints included

### 3. Daily to monthly, prices to returns

In [ ]:
monthly = spy["adj_close"].resample("ME").last()
m_rets = monthly.pct_change(fill_method=None)
m_rets.head(3)

The first `NaN` is correct (no month before the first month). `fill_method=None` means missing prices stay missing — nothing is silently filled.

### 4. One picture

In [ ]:
g = (1 + m_rets.fillna(0)).cumprod()
ax = g.plot(figsize=(8, 3.5), color="#0F6BBA",
            xlabel="", ylabel="Value of £1",
            title="SPY: growth of £1 since 2010")

That pipeline — filter, index, resample, `pct_change`, `cumprod`, plot — is this week in five lines. Everything below drives it harder: on the raw file, on a dirty column, on a client's question, and on an AI's account of what the code does.

---

### 5. Warm-up: interrogate the file

Three questions get put to `stocks_daily.parquet` before anything is computed from it: how many rows there are and what period they span, how those rows split across tickers, and whether one fact we can check by hand — NVDA's highest traded price — comes out where we expect. One or two lines each.

In [ ]:
# (a)
len(stocks), stocks["date"].min(), stocks["date"].max()

In [ ]:
# (b)
stocks["ticker"].value_counts()

SPY and the individual stocks should have essentially the same count (they trade on the same calendar); small differences would be a data-quality flag worth investigating — next week's topic.

In [ ]:
# (c)
nvda = stocks.loc[stocks["ticker"] == "NVDA"]
nvda.loc[nvda["close"].idxmax(), ["date", "close"]]

`idxmax()` returns the *index label* of the maximum, which we then feed to `.loc` to recover the whole row. Note we used `close` rather than `adj_close`: `adj_close` also folds dividends back into the history, so the two columns sit at different levels in earlier years and their maxima need not fall on the same date. (In this file both columns are already adjusted for stock splits — NVDA split 4-for-1 in 2021 and 10-for-1 in 2024 — so neither series jumps on a split date.) Knowing *which* price column answers *which* question is a Week 03 obsession, seeded here.

---

### 6. Predict, then run

Four expressions on a four-row frame, chosen because two of them return the same answer for opposite reasons. Read down them and settle on what each one gives *before* the cell below runs them — the gap between your answer and pandas' is the thing worth finding.

```python
df = pd.DataFrame({"ret": [0.01, -0.02, 0.03, -0.04]},
                  index=pd.to_datetime(["2024-01-31", "2024-02-29",
                                        "2024-03-31", "2024-04-30"]))
df.iloc[1:3]                          # (a) how many rows?
df.loc["2024-02-29":"2024-03-31"]     # (b) how many rows?
df.iloc[-1]                           # (c) which month?
df.loc["2024"]                        # (d) how many rows?
```

In [ ]:
df = pd.DataFrame({"ret": [0.01, -0.02, 0.03, -0.04]},
                  index=pd.to_datetime(["2024-01-31", "2024-02-29",
                                        "2024-03-31", "2024-04-30"]))
print(len(df.iloc[1:3]), len(df.loc["2024-02-29":"2024-03-31"]),
      df.iloc[-1].name.month_name(), len(df.loc["2024"]))

- (a) **2 rows** (positions 1 and 2) — positional slicing is half-open.
- (b) **2 rows** — but for the opposite reason: label slicing includes *both* endpoints, February and March.
- (c) **April** — `iloc[-1]` is the last row.
- (d) **4 rows** — a partial string label selects everything within that period.

(a) and (b) returning the same count for different reasons is exactly why you must always know which accessor you are using: the count alone never tells you whether the rule you applied was the right one.

---

### 7. The object-dtype repair drill

A vendor file arrives with volume stored as text, including thousands separators and a stray placeholder:

```python
vol = pd.Series(["1,204,300", "998,100", "-", "1,455,800", "1,002,650"])
```

We do three things with it, in this order: ask the text Series for its maximum and watch what comes back; locate the value that cannot be converted to a number; and only then produce a clean `float64` Series, stating explicitly what became of the placeholder.

In [ ]:
vol = pd.Series(["1,204,300", "998,100", "-", "1,455,800", "1,002,650"])
# (a)
vol.max()

**(a)** These are strings, so `.max()` compares them alphabetically, character by character: `"9"` beats `"1"`, so `"998,100"` — the *smallest* volume — is returned as the "maximum". No error is raised. This is the quiet failure from the lecture, and it is why the dtype check is unconditional.

**(b)** Strip the separators first, then use `errors="coerce"` as a detector:

In [ ]:
stripped = vol.str.replace(",", "", regex=False)
coerced = pd.to_numeric(stripped, errors="coerce")
vol[coerced.isna()]

Only the placeholder `"-"` fails; the comma-laden values convert fine once the separators are removed.

**(c)** Having *seen* that the only casualty is a genuine missing-value marker, coercing it to `NaN` is the right call — and we say so in the code:

In [ ]:
clean = pd.to_numeric(stripped, errors="coerce")  # "-" is missing volume -> NaN
clean

In [ ]:
clean.dtype, clean.max()

Now `.max()` returns the true maximum. The workflow — inspect what fails, *then* decide — is the point; `errors="coerce"` applied blindly makes data disappear without a trace.

---

### 8. SPY's best and worst months

SPY's monthly return series over the full sample — most of it built in the demo above — and then the two dates at its extremes. The ground truth to land on: both sit in **2020**, the worst in March (about −13%), the best in April (about +13%).

In [ ]:
spy = (stocks.loc[stocks["ticker"] == "SPY", ["date", "adj_close"]]
       .set_index("date").sort_index())
m_rets = spy["adj_close"].resample("ME").last().pct_change(fill_method=None)

# (a)
best, worst = m_rets.idxmax(), m_rets.idxmin()
print("best :", best.date(), f"{m_rets.max():+.4f}")
print("worst:", worst.date(), f"{m_rets.min():+.4f}")

**(b)** The worst month ends 2020-03-31 at roughly **−0.13** and the best ends 2020-04-30 at roughly **+0.13** — the COVID crash and the immediate rebound, back to back. We encode that check as assertions rather than eyeballing it:

In [ ]:
assert worst == pd.Timestamp("2020-03-31"), "worst month should be March 2020"
assert best == pd.Timestamp("2020-04-30"), "best month should be April 2020"
print("ground truth verified")

The symmetry is a small trap worth savouring: −13% followed by +13% does **not** take you back to where you started (0.87 × 1.13 ≈ 0.98). Compounding is not addition — Week 07's performance metrics build on exactly this.

**(c)** The first return is `NaN` because no prior month-end price exists — it is the mathematically correct answer, not a defect. Keep it, and convert it (e.g. `fillna(0)`, "no return earned before you were invested") only at the point of use, explicitly, so the assumption is visible in your code rather than buried in a cleaning step.

---

### 9. Growth of £1: JPM vs XOM vs the market

A client asks: "Over 2010–2019, would I have done better in JP Morgan, Exxon, or just the market (SPY)?"

Answering it takes three moves: growth of £1 for the three tickers over that fixed window (`adj_close`, with `fill_method=None`); one matplotlib chart of the three lines on a log scale, properly labelled; and one sentence back to the client with numbers in it. The sanity checks: SPY ≈ £3.5, XOM well under £2, JPM ahead of SPY.

In [ ]:
sub = stocks[stocks["ticker"].isin(["JPM", "XOM", "SPY"])]
prices = sub.pivot(index="date", columns="ticker", values="adj_close")
window = prices.loc["2010":"2019"]
growth = (1 + window.pct_change(fill_method=None).fillna(0)).cumprod()
growth.iloc[-1].sort_values(ascending=False)

The pipeline is the lecture's, restricted to three tickers and the fixed window. JPM finishes ahead of SPY (roughly £4 versus ≈ £3.5), with XOM far behind at under £1.5 — an oil major that essentially sat out the decade.

In [ ]:
ax = growth.plot(logy=True, figsize=(8, 3.8),
                 xlabel="", ylabel="Value of £1 (log scale)",
                 title="Growth of £1, 2010–2019")

A defensible sentence for the client: "£1 invested at the start of 2010 grew to roughly £4 in JPM, about £3.5 in SPY, and under £1.5 in XOM by end-2019 — JPM beat the market, Exxon badly lagged it." Note what you are *not* yet entitled to say: that JPM was the better *choice*. It may simply have been riskier — the risk-adjustment machinery arrives in Weeks 04–07. And picking winners after watching them win is survivorship-flavoured hindsight, villain number two from Week 01.

---

### 10. Verify the AI: check its explanation

You will often paste code you did not write into an assistant and ask "explain this line by line". The step everyone skips is the next one — *checking the explanation* — and that is what this section does.

The code (it is correct and runs fine):

```python
spy = stocks.loc[stocks["ticker"] == "SPY", ["date", "adj_close"]]
spy = spy.set_index("date").sort_index()
monthly = spy["adj_close"].resample("ME").last()
window = monthly.loc["2010":"2019"]
rets = window.pct_change(fill_method=None)
worst = rets.min()
```

An AI assistant produced this explanation (cached here so everyone works from the same text; ask your own assistant the same question and compare if you like):

> 1. *Line 1 filters the table to SPY rows only and keeps just the `date` and `adj_close` columns.*
> 2. *Line 2 makes `date` the index and sorts it — required for time-based operations like resampling.*
> 3. *Line 3 groups the daily prices by calendar month and takes the last available price in each month, giving a month-end price series.*
> 4. *Line 4 slices the series to the study window. Because Python slicing is half-open, this keeps months from January 2010 up to but not including 2019 — the last month retained is December 2018.*
> 5. *Line 5 computes month-on-month simple returns; the first value is `NaN` since there is no earlier month, and `fill_method=None` stops pandas from forward-filling gaps in the prices.*
> 6. *Line 6 extracts the most negative monthly return in the window.*

Exactly **one** numbered claim is wrong. Commit to a candidate before reading on — the habit being built is suspecting a specific line and then settling it with code, and two lines are enough to settle this one.

Claim **4** is wrong. The other five are accurate — which is precisely what makes this dangerous: a fluent, 83%-correct explanation invites you to trust the remaining 17%.

The evidence comes from running the code and interrogating the window:

In [ ]:
spy_q6 = stocks.loc[stocks["ticker"] == "SPY", ["date", "adj_close"]]
spy_q6 = spy_q6.set_index("date").sort_index()
monthly_q6 = spy_q6["adj_close"].resample("ME").last()
window_q6 = monthly_q6.loc["2010":"2019"]
len(window_q6), window_q6.index[0].date(), window_q6.index[-1].date()

120 months, ending **2019-12-31**. If the assistant were right, there would be 108 months ending 2018-12-31. Two lines of evidence beat any amount of debate.

The corrected sentence reads: "*Line 4 slices by index* ***label***: `.loc` *slicing is inclusive of both endpoints, and a partial label like* `"2019"` *covers the whole of that year — so the window runs from January 2010 to December 2019 inclusive, 120 months.*" Half-open slicing is real, but it belongs to *positional* indexing (`iloc` and plain lists) — the assistant applied a true rule in the wrong place, the most common species of LLM error you will meet in this unit.

**Log it.** Prompt, output, evidence, correction — that is the whole record of a caught AI error, and it takes a minute to write down. Keep an AI log of your own — an `ai_log.md` file or a Google Doc in your Drive is enough — and paste this one in as entry zero.

---

### 11. Assertions first: `annualise` (stretch)

The lecture's closing activity, finished here. The spec:

> `annualise(monthly_ret)` takes one *monthly* simple return (a float, e.g. `0.01`) and returns the annualised simple return.

The order is the whole point: two assertions first — one of which has to catch a lazy implementation returning `monthly_ret * 12` — and only then the one-line implementation, run against them.

Annualising a simple return compounds it: $(1 + r_m)^{12} - 1$. Two hand-checkable cases:

In [ ]:
def check(f):
    assert f(0.0) == 0.0, "zero compounds to zero"
    assert abs(f(0.1) - (1.1**12 - 1)) < 1e-12, "10% monthly compounds to ~213.8% annually"
    return "assertions pass"

The second case is the killer: `0.1 * 12 = 1.2`, whereas $(1.1)^{12} - 1 \approx 2.138$ — a 94-percentage-point discrepancy no rounding excuse can absorb. (The first case alone would *not* catch the `*12` cheat, since $0 \times 12 = 0$ too — assertions must include a non-trivial case.)

The implementation, written second:

In [ ]:
def annualise(monthly_ret):
    """Annualised simple return from one monthly simple return."""
    return (1 + monthly_ret) ** 12 - 1

check(annualise)

And the same tests turned on the lazy version, to confirm they bite:

In [ ]:
check(lambda r: r * 12)

The assertion fires, as designed. Compounding-versus-multiplying is not a toy distinction: annualisation constants (×12, ×√12, ×252, ×√252) appear throughout Week 07's performance metrics, and mixing them up is one of the three bugs planted in that week's rigged backtest. You have now built the tool that catches it.

#### ⚠️ Common mistakes this week
1. **Off-by-one slices** — expecting `x[1:3]` to return three elements or `x[1]` to be the first. Symptom: silently wrong windows, no error message.
2. **`.loc`/`.iloc` endpoint confusion** — label slices include both endpoints; positional slices exclude the stop. The *Predict, then run* section exists because both give "2 rows" for opposite reasons.
3. **Skipping the dtype check** — analysis "works" on an `object` column and returns confident nonsense (`"9" > "100"`). Run `df.dtypes` after *every* read.
4. **Filling gaps before looking** — `fillna(0)` or default fill methods applied upstream manufacture returns out of missing data. Keep `pct_change(fill_method=None)`; make filling decisions explicitly, at the point of use.
5. **Out-of-order cells and a misplaced data folder** — cell 12 works only because cell 7 ran an hour ago, or the Setup cell stops with "Data folder not found" because `data` is not at `MyDrive/ACFI20009/data`. Habit: read data through `DATA`, and use Runtime → Restart and run all before believing any result.